In [ ]:
import numpy as np # libreria per operazioni tra matrici e vettori
import tensorflow as tf # libreria di riferimento per il DL
from tensorflow import keras # fornisce interfaccia semplificate rispetto a tensorflow


In [ ]:
data_path = keras.utils.get_file(
    "news20.tar.gz",
    "http://www.cs.cmu.edu/afs/cs.cmu.edu/project/theo-20/www/data/news20.tar.gz",
    untar=True,
) # get_file mi permette di scaricare un file (il mio dataset in questo caso)

17329808/17329808 ━━━━━━━━━━━━━━━━━━━━ 30s 2us/step


In [ ]:
import os
import pathlib

data_dir = pathlib.Path(data_path) / "20_newsgroup"
dirnames = os.listdir(data_dir)
print("Number of directories:", len(dirnames))
print("Directory names:", data_dir)

fnames = os.listdir(data_dir / "comp.graphics")
print("Number of files in comp.graphics:", len(fnames))
print("Some example filenames:", fnames[:5])

Number of directories: 20
Directory names: /root/.keras/datasets/news20_extracted/20_newsgroup
Number of files in comp.graphics: 1000
Some example filenames: ['38286', '38848', '39027', '38221', '38890']


In [ ]:
print(open(data_dir / "comp.graphics" / "38254").read()) # stampo un esempio del dataset dalla cartella comp.graphics

Newsgroups: comp.graphics
Path: cantaloupe.srv.cs.cmu.edu!crabapple.srv.cs.cmu.edu!fs7.ece.cmu.edu!europa.eng.gtefsd.com!news.ans.net!malgudi.oar.net!zaphod.mps.ohio-state.edu!uunet.ca!canrem!dosgate!dosgate![danny.hawrysio@canrem.com]
From: "danny hawrysio" <danny.hawrysio@canrem.com>
Subject: windows imagine??!!
Message-ID: <199314.4387.24074@dosgate>
Reply-To: "danny hawrysio" <danny.hawrysio@canrem.com>
Organization: Canada Remote Systems
Distribution: comp
Date: 14 Apr 93 14:45:43 EST
Lines: 12


-> I have been on the phone with Impulse for about 3 months waiting for
-> my cross - platform upgrade (Amiga to IBM). They have told me every
-> week for 3 months, "it will be ready next week". Still waiting.......

 They've been saying that for two years now, you'd think by now people
wouldn't go on about it being 'soon' and only believe it when they can
buy it. I wish Amiga users wouldn't be so gullible (gasp, how dare he
say that!).
--
Canada Remote Systems - Toronto, Ontario
416-629-

In [ ]:
# definisco le strutture dati
# in samples metto ogni esempio del dataset
# in labels metto le categorie

samples = []
labels = []
class_names = []
class_index = 0
for dirname in sorted(os.listdir(data_dir)):
    class_names.append(dirname)
    dirpath = data_dir / dirname
    fnames = os.listdir(dirpath)
    print("Processing %s, %d files found" % (dirname, len(fnames)))
    for fname in fnames:
        fpath = dirpath / fname
        f = open(fpath, encoding="latin-1")
        content = f.read()
        lines = content.split("\n")
        lines = lines[10:] # parte dalla decima riga perché prendo solo il body del messaggio (le prime 10 righe sono intestazione
        content = "\n".join(lines)
        samples.append(content)
        labels.append(class_index)
    class_index += 1

print("Classes:", class_names)
print("Number of samples:", len(samples))
print("Number of labels:", len(labels))

Processing alt.atheism, 1000 files found
Processing comp.graphics, 1000 files found
Processing comp.os.ms-windows.misc, 1000 files found
Processing comp.sys.ibm.pc.hardware, 1000 files found
Processing comp.sys.mac.hardware, 1000 files found
Processing comp.windows.x, 1000 files found
Processing misc.forsale, 1000 files found
Processing rec.autos, 1000 files found
Processing rec.motorcycles, 1000 files found
Processing rec.sport.baseball, 1000 files found
Processing rec.sport.hockey, 1000 files found
Processing sci.crypt, 1000 files found
Processing sci.electronics, 1000 files found
Processing sci.med, 1000 files found
Processing sci.space, 1000 files found
Processing soc.religion.christian, 997 files found
Processing talk.politics.guns, 1000 files found
Processing talk.politics.mideast, 1000 files found
Processing talk.politics.misc, 1000 files found
Processing talk.religion.misc, 1000 files found
Classes: ['alt.atheism', 'comp.graphics', 'comp.os.ms-windows.misc', 'comp.sys.ibm.pc.ha

In [ ]:
# creazione dei vettori tramite la classe tokenizer

from tensorflow.keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical


tokenizer = Tokenizer(num_words=20000,oov_token=True)
tokenizer.fit_on_texts(samples)
sequences = tokenizer.texts_to_sequences(samples)

word_index = tokenizer.word_index
print('Found %s unique tokens.' % len(word_index))

data = pad_sequences(sequences, maxlen=1000) # applico il padding -> esempi con più di 1000 token verranno troncati, quelli con meno verranno aggiunti
label = to_categorical(np.asarray(labels)) # converte un vettore di interi in una matrice binaria, usato per le funzioni di tipo categorical cross-entropy (boh), praticamente ora ogni riga della matrice è un array di 0 e 1 a seconda della categoria del documento

print('Shape of data tensor:', data.shape)
print('Shape of label tensor:', label.shape)


Found 175747 unique tokens.
Shape of data tensor: (19997, 1000)
Shape of label tensor: (19997, 20)


In [ ]:
# split the data into a training set and a validation set

validation_split = 0.2

indices = np.arange(data.shape[0])
np.random.shuffle(indices)
data = data[indices]
labels = label[indices]
nb_validation_samples = int(validation_split * data.shape[0])

x_train = data[:-nb_validation_samples]
y_train = labels[:-nb_validation_samples]
x_val = data[-nb_validation_samples:]
y_val = labels[-nb_validation_samples:]

In [ ]:
!wget https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip
!unzip -q glove.6B.zip

--2026-05-27 12:17:40--  https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip
Resolving downloads.cs.stanford.edu (downloads.cs.stanford.edu)... 171.64.64.22
Connecting to downloads.cs.stanford.edu (downloads.cs.stanford.edu)|171.64.64.22|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 862182613 (822M) [application/zip]
Saving to: ‘glove.6B.zip’

glove.6B.zip        100%[===================>] 822.24M  4.91MB/s    in 2m 47s  

2026-05-27 12:20:29 (4.91 MB/s) - ‘glove.6B.zip’ saved [862182613/862182613]



In [ ]:
#Preparing the Embedding layer (input layer)

EMBEDDING_DIM = 100
embeddings_index = {}

f = open('glove.6B.100d.txt')
for i, line in enumerate(f):
    values = line.strip().split()
    if len(values) != EMBEDDING_DIM + 1:  # parola + 100 valori
        print(f"Riga {i} problematica:", values[:5])
        continue  # salta righe corrotte o vuote
    word = values[0]
    coefs = np.asarray(values[1:], dtype='float32')
    embeddings_index[word] = coefs
f.close()

print('Found %s word vectors.' % len(embeddings_index))

Found 400000 word vectors.


In [ ]:
embedding_matrix = np.zeros((len(word_index) + 1, EMBEDDING_DIM))
for word, i in word_index.items():
    embedding_vector = embeddings_index.get(word)
    if embedding_vector is not None:
        # words not found in embedding index will be all-zeros.
        embedding_matrix[i] = embedding_vector

In [ ]:
print (embeddings_index.get('apple'))

[-0.5985    -0.46321    0.13001   -0.019576   0.4603    -0.3018
  0.8977    -0.65634    0.66858   -0.49164    0.037557  -0.050889
  0.6451    -0.53882   -0.3765    -0.04312    0.51384    0.17783
  0.28596    0.92063   -0.49349   -0.48583    0.61321    0.78211
  0.19254    0.91228   -0.055596  -0.12512   -0.65688    0.068557
  0.55629    1.611     -0.0073642 -0.48879    0.45493    0.96105
 -0.063369   0.17432    0.9814    -1.3125    -0.15801   -0.54301
 -0.13888   -0.26146   -0.3691     0.26844   -0.24375   -0.19484
  0.62583   -0.7377     0.38351   -0.75004   -0.39053    0.091498
 -0.36591   -1.4715    -0.45228    0.2256     1.1412    -0.38526
 -0.06716    0.57288   -0.39191    0.31302   -0.29235   -0.96157
  0.15154   -0.21659    0.25103    0.096967   0.2843     1.4296
 -0.50565   -0.51374   -0.47218    0.32036    0.023149   0.22623
 -0.09725    0.82126    0.92599   -1.0086    -0.38639    0.86408
 -1.206     -0.28528    0.2265    -0.38773    0.40879    0.59303
  0.30769    0.83804   -

In [ ]:
from keras.layers import Embedding

# EMBEDDING_DIM=100 -- tanto è definita sopra
MAX_SEQUENCE_LENGTH=200
embedding_layer = Embedding(len(word_index) + 1,
                            EMBEDDING_DIM,
                            weights=[embedding_matrix], # se non lo metto mi addestra lui il layer che viene addestrato man mano durante il training (è un approccio che overfitta sui dati di training)
                            # input_length=MAX_SEQUENCE_LENGTH, # è un parametro deprecato
                            trainable=False) # per fare in modo che non cambino i pesi della matrice durante l'addestramento, altrimenti metto True, così addestro sul task specifico

In [ ]:
#Build the model

from keras.layers import Input
from keras.layers import Conv1D
from keras.layers import MaxPooling1D
from keras.layers import GlobalMaxPooling1D
from keras.layers import Dense
from keras.layers import Dropout
from keras import Model

MAX_SEQUENCE_LENGTH=1000
sequence_input = Input(shape=(MAX_SEQUENCE_LENGTH,), dtype='int32')
embedded_sequences = embedding_layer(sequence_input)
x = Conv1D(128, 5, activation='relu')(embedded_sequences) # 128 n. filtri in output, 5 finestra del kernel size, funzione di attivazione relu, input
x = MaxPooling1D(5)(x) # riduce la dimensione delle matrici in input
x = Conv1D(128, 5, activation='relu')(x)
x = GlobalMaxPooling1D()(x)  # global max pooling
x = Dropout(0.5)(x) # riduce l'overfitting, droppo il 50% dei neuroni per forzare i pesi ad essere equamente distribuiti
x = Dense(128, activation="relu")(x) # layer che recupera info non influenzate dopo il dropout
preds = Dense(len(class_names), activation="softmax")(x) # softmax calcola una probabilità per ogni singola classe
model = Model(sequence_input, preds)
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 1000)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 1000, 100)      │    17,574,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 996, 128)       │        64,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 199, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 195, 128)       │        82,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d            │ (None, 128)            │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 20)             │         2,580 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 17,740,068 (67.67 MB)

 Trainable params: 165,268 (645.58 KB)

 Non-trainable params: 17,574,800 (67.04 MB)

In [ ]:
model.compile(loss='categorical_crossentropy', optimizer='adam',metrics=['acc'])

history = model.fit(x_train, y_train, validation_data=(x_val, y_val),
                    epochs=100, batch_size=64) # posso aumentare il numero di batch size o la lunghezza massima dei documenti per aumentare l'accuracy, se poi metto altri layer convoluzionali è meglio (devo sempre mettere un layer di maxpooling)

Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 23s 47ms/step - acc: 0.1388 - loss: 2.6800 - val_acc: 0.3038 - val_loss: 1.9374
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 23s 15ms/step - acc: 0.3650 - loss: 1.8079 - val_acc: 0.4929 - val_loss: 1.4083
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 4s 17ms/step - acc: 0.5146 - loss: 1.3976 - val_acc: 0.6332 - val_loss: 1.1237
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 4s 16ms/step - acc: 0.6042 - loss: 1.1533 - val_acc: 0.6597 - val_loss: 1.0076
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 4s 15ms/step - acc: 0.6585 - loss: 0.9910 - val_acc: 0.6672 - val_loss: 0.9879
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 4s 15ms/step - acc: 0.6951 - loss: 0.8977 - val_acc: 0.6979 - val_loss: 0.8976
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 4s 16ms/step - acc: 0.7270 - loss: 0.8008 - val_acc: 0.7117 - val_loss: 0.8710
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 4s 16ms/step - acc: 0.7497 - loss: 0.7265 - val_acc: 0.7217 - val_loss: 0.8483
Epoch 9/100
250/250 ━━━━━━━━━━━━━━━━━━

In [ ]:
labels = np.array(['alt.atheism', 'comp.graphics', 'comp.os.ms-windows.misc',
 'comp.sys.ibm.pc.hardware', 'comp.sys.mac.hardware', 'comp.windows.x',
 'misc.forsale', 'rec.autos', 'rec.motorcycles', 'rec.sport.baseball',
 'rec.sport.hockey', 'sci.crypt', 'sci.electronics', 'sci.med', 'sci.space',
 'soc.religion.christian', 'talk.politics.guns', 'talk.politics.mideast',
 'talk.politics.misc', 'talk.religion.misc'])

x_data = []
x_data.append("""
Path: cantaloupe.srv.cs.cmu.edu!crabapple.srv.cs.cmu.edu!fs7.ece.cmu.edu!news.sei.cmu.edu!cis.ohio-state.edu!bounce-bounce
From: myoakam@cis.ohio-state.edu (micah r yoakam)
Newsgroups: misc.forsale
Subject: BOAT for SALE
Date: 31 Mar 1993 11:38:38 -0500
Organization: The Ohio State University Dept. of Computer and Info. Science
Lines: 14
Distribution: USA
Expires: +60days
Message-ID: <1pcheeINN3i6@eucalyptus.cis.ohio-state.edu>
NNTP-Posting-Host: eucalyptus.cis.ohio-state.edu

BOAT For SALE
1989 23' IMPERIAL FISHERMAN featuring
        Walkaround Cuddy Cabin, 305 V8 with VOLVO DUO PROP OUTDRIVE /\/\/\/
AM-FM Cassette Stereo, VHF RADIO, 4x6 HUMMINGBIRD Fishfinder, ALL  Safty
equipment, Covers, and MUCH MORE.
        18000 LB.  Capacity
        includes Storage Trailer
        Hardly used:  LESS Than 100 Hrs

Asking: $15,000 OR Best OFFER.
For Further information contact Gerald at 1-(419)-756-2950
                                        Mansfield, OH
""")


x_tokenized = tokenizer.texts_to_sequences(x_data)
x_pad = pad_sequences(x_tokenized, maxlen=1000)



i=0
for x_t in x_pad:
    prediction = model.predict(np.array([x_t]))
    predicted_label = labels[np.argmax(prediction[0])]
    print(prediction)
    print("Input Text", "Predicted label: " + predicted_label)
    i += 1

<>:24: SyntaxWarning: invalid escape sequence '\/'
<>:24: SyntaxWarning: invalid escape sequence '\/'
/tmp/ipykernel_3497/1945029157.py:24: SyntaxWarning: invalid escape sequence '\/'
  Walkaround Cuddy Cabin, 305 V8 with VOLVO DUO PROP OUTDRIVE /\/\/\/


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 726ms/step
[[1.4469423e-20 4.1517167e-13 1.0965246e-15 1.9722480e-08 6.6530279e-09
  1.9441600e-17 9.9821043e-01 1.5601984e-03 9.3275304e-07 2.2699880e-11
  2.0390614e-12 4.2531374e-14 2.2846037e-04 3.7790040e-14 1.0407428e-10
  4.0138079e-25 2.8684488e-12 2.8598930e-20 9.3492098e-16 8.7057358e-19]]
Input Text Predicted label: misc.forsale
